# 🧹 Remove `â` Artifacts
Reads the cleaned CSV, replaces all `â` characters (and their trailing junk bytes) with an empty string, then exports a new clean CSV.

## 1. 📦 Import Libraries

In [1]:
import pandas as pd
import re
from pathlib import Path

print(f"pandas version : {pd.__version__}")

pandas version : 2.2.3


## 2. 📂 Load CSV

In [2]:
INPUT_FILE  = "liste_des_competences_clean.csv"
OUTPUT_FILE = "liste_des_competences_no_artifacts.csv"

assert Path(INPUT_FILE).exists(), f"File not found: {INPUT_FILE}"

df = pd.read_csv(INPUT_FILE, dtype=str, encoding="utf-8-sig")

print(f"✅ Loaded  {INPUT_FILE}")
print(f"   Shape  : {df.shape[0]} rows × {df.shape[1]} columns")

✅ Loaded  liste_des_competences_clean.csv
   Shape  : 158 rows × 12 columns


## 3. 🔍 Inspect â Occurrences

In [3]:
# Pattern: â followed by any non-space characters (encoding artifact)
ARTIFACT_PATTERN = re.compile(r'â\S*')

mask = df.apply(lambda col: col.astype(str).str.contains('â', na=False))
total = mask.values.sum()

print(f"🔎 Cells containing 'â' artifacts : {total}")
print()
print("Per-column breakdown:")
col_counts = mask.sum().rename("Cells with â")
display(col_counts[col_counts > 0].to_frame())

🔎 Cells containing 'â' artifacts : 95

Per-column breakdown:


,Cells with â
Competence,3
Indicateurs_Observables,1
Preuves_Attendues,6
Situations_Professionnelles_Type,1
Outils_SI_ou_Materiels,40
Reglementation_ou_Normes,43
Mots_Cles,1


## 4. 🧹 Remove â Artifacts

In [4]:
def remove_artifacts(value: object) -> object:
    """
    Remove all â artifact sequences from a cell value.
    - Replaces â + trailing non-space chars with empty string
    - Cleans up leftover double spaces
    - Preserves NaN and non-string values unchanged
    """
    if pd.isna(value):
        return value
    if not isinstance(value, str):
        return value
    if 'â' not in value:
        return value                              # skip clean cells

    cleaned = ARTIFACT_PATTERN.sub('', value)    # remove â + junk chars
    cleaned = re.sub(r' {2,}', ' ', cleaned)     # collapse double spaces
    cleaned = cleaned.strip()                    # trim leading/trailing spaces
    return cleaned


# Apply vectorized across all columns
df_clean = df.apply(lambda col: col.map(remove_artifacts))

# Verify
remaining = df_clean.apply(
    lambda col: col.astype(str).str.contains('â', na=False)
).values.sum()

print(f"✅ Done. Remaining 'â' cells : {remaining}")

✅ Done. Remaining 'â' cells : 0


## 5. 🔄 Before / After Sample

In [5]:
changes = []
for col in df.columns:
    before = df[col].astype(str)
    after  = df_clean[col].astype(str)
    diff   = before[before != after].index
    for idx in diff:
        changes.append({
            "Row"    : idx,
            "Column" : col,
            "BEFORE" : df.at[idx, col],
            "AFTER"  : df_clean.at[idx, col],
        })

diff_df = pd.DataFrame(changes)
print(f"Total cells modified : {len(diff_df)}")
print()
pd.set_option("display.max_colwidth", 80)
display(diff_df.head(20))

Total cells modified : 95



,Row,Column,BEFORE,AFTER
0,27,Competence,"Soutenir lâeffort et le stress (port, posture, charge mentale)","Soutenir l et le stress (port, posture, charge mentale)"
1,31,Competence,Travailler en autonomie avec calme et gestion dâespace restreint,Travailler en autonomie avec calme et gestion d restreint
2,97,Competence,Optimiser la surface et lâimplantation stockage,Optimiser la surface et l stockage
3,47,Indicateurs_Observables,"Taux dâerreur faible, confirmations correctes, rythme stable","Taux d faible, confirmations correctes, rythme stable"
4,43,Preuves_Attendues,Plan dâimplantation + KPI,Plan d + KPI
5,57,Preuves_Attendues,Plan dâaction,Plan d
6,78,Preuves_Attendues,Plan dâanimation,Plan d
7,91,Preuves_Attendues,Note dâarbitrage,Note d
8,98,Preuves_Attendues,Plan dâaction,Plan d
9,109,Preuves_Attendues,Plan dâimplantation,Plan d


## 6. 💾 Export

In [6]:
df_clean.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",   # BOM → opens correctly in Excel
    sep=","
)

size_kb = Path(OUTPUT_FILE).stat().st_size / 1024
print(f"✅ Exported → {OUTPUT_FILE}")
print(f"   Rows      : {len(df_clean)}")
print(f"   Columns   : {len(df_clean.columns)}")
print(f"   Size      : {size_kb:.1f} KB")
print(f"   Encoding  : UTF-8 with BOM")
print(f"   Modified  : {len(diff_df)} cells cleaned")

✅ Exported → liste_des_competences_no_artifacts.csv
   Rows      : 158
   Columns   : 12
   Size      : 44.0 KB
   Encoding  : UTF-8 with BOM
   Modified  : 95 cells cleaned
